<a href="https://colab.research.google.com/github/Dani2003/paper-implementations/blob/main/notebooks/01_LoRA_From_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Re-Implementing LoRA From Scratch**

This notebook reconstructs the core ideas presented in Microsoft's LoRA: Low-Rank Adaptation of Large Language Models paper without relying on PEFT or the official implementation. The objective is not to reproduce benchmark results, but to understand the mathematical formulation, engineering decisions, and parameter-efficient fine-tuning strategy introduced by the authors.

Throughout the notebook, LoRA is implemented incrementally—from building a custom low-rank layer, to integrating it into GPT-2, and finally verifying that only the adapter parameters participate in optimization.

1. Verifying the Compute Environment

Before implementing LoRA, we first verify that PyTorch can access a CUDA-enabled GPU. Although LoRA dramatically reduces the number of trainable parameters, transformer models such as GPT-2 still contain millions of frozen parameters that must participate in the forward pass. Running these computations on a GPU significantly accelerates experimentation and training.

This cell checks whether CUDA is available and prints the name of the detected GPU. Confirming the execution environment at the beginning of the notebook also improves reproducibility, since hardware can influence training speed and memory usage.

In [2]:
import torch
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

GPU Available: True
Device Name: Tesla T4


2. Implementing a Custom LoRA Linear Layer

Instead of relying on existing libraries such as PEFT, this implementation recreates the core LoRA algorithm directly in PyTorch. The objective is to understand how Low-Rank Adaptation modifies a frozen neural network while training only a small number of additional parameters.

The implementation begins by constructing a standard linear layer that represents the frozen pre-trained weights W0. Two learnable low-rank matrices, A and B, are then introduced to approximate an update matrix

ΔW=BA

where the rank r is intentionally much smaller than the original weight dimensions.

Following the initialization strategy proposed in the LoRA paper, matrix A is initialized using Kaiming Uniform initialization while matrix B is initialized to zeros. As a result,

ΔW=0

at the beginning of training, ensuring that the network initially behaves exactly like the original pre-trained model.

During the forward pass, the frozen output

W0​x

is combined with the scaled low-rank update

rα​BAx

where the scaling factor controls the magnitude of the adaptation without modifying the frozen weights.

By freezing the original parameters and optimizing only the low-rank matrices, the number of trainable parameters is reduced substantially while preserving the knowledge stored in the pre-trained model.

In [4]:
import torch
import torch.nn as nn
import math

class CustomLoRALinear(nn.Module):
    def __init__(self, in_features, out_features, r=8, lora_alpha=16):
        super(CustomLoRALinear, self).__init__()

        # 1. The original base linear layer (representing our frozen pre-trained model)
        self.linear = nn.Linear(in_features, out_features)

        # 2. Hyperparameters for rank and scaling factor
        self.r = r
        self.lora_alpha = lora_alpha
        self.scaling = lora_alpha / r

        # 3. Only instantiate LoRA matrices if rank > 0
        if r > 0:
            # Matrix A down-projects from in_features to r
            self.lora_A = nn.Parameter(torch.zeros((r, in_features)))
            # Matrix B up-projects from r to out_features
            self.lora_B = nn.Parameter(torch.zeros((out_features, r)))

            # 4. Critical initialization strategy from the paper:
            # Kaiming uniform for A, and zeros for B so that Delta W starts at 0!
            nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
            nn.init.zeros_(self.lora_B)

        # 5. Freeze the core base weights
        self.linear.weight.requires_grad = False

    def forward(self, x):
        # Base layer forward pass: W0 * x
        base_output = self.linear(x)

        if self.r > 0:
            # LoRA forward pass equation: h = W0*x + ((alpha/r) * B * A * x)
            # We use matrix multiplication (@) and transpose weights to align dimensions
            lora_output = (x @ self.lora_A.t() @ self.lora_B.t()) * self.scaling
            return base_output + lora_output

        return base_output

In [5]:
# Create dummy input: Batch size = 2, Sequence length = 5, Input dimensions = 128
dummy_input = torch.randn(2, 5, 128).cuda()

# Initialize our custom LoRA layer mapping 128 dims to 64 dims with rank r=4
lora_layer = CustomLoRALinear(in_features=128, out_features=64, r=4).cuda()

# Pass data through the layer
output = lora_layer(dummy_input)

print("Input shape:", dummy_input.shape)
print("Output shape:", output.shape)
print("Are base linear weights frozen?", not lora_layer.linear.weight.requires_grad)
print("Is lora_A trainable?", lora_layer.lora_A.requires_grad)

Input shape: torch.Size([2, 5, 128])
Output shape: torch.Size([2, 5, 64])
Are base linear weights frozen? True
Is lora_A trainable? True


4. Injecting LoRA into GPT-2 via Model Surgery

After validating the custom LoRA layer in isolation, the next step is to integrate it into a real transformer architecture. Rather than modifying GPT-2's source code directly, this notebook performs model surgery by traversing the model hierarchy and selectively replacing attention projection layers with custom LoRA-augmented layers.

The apply_lora_surgery() function recursively scans the model and identifies target modules-in this implementation, the c_attn projection used by GPT-2's self-attention mechanism. For each matching layer, a new CustomLoRALinear module is created with the same input and output dimensions.

To preserve the behavior of the original pre-trained model, the existing GPT-2 weights and biases are copied into the new layer before replacement. Since Hugging Face stores GPT-2 attention projections using a Conv1D representation with transposed weights, the weight matrix is transposed before being copied into the standard PyTorch linear layer.

Finally, the original module is replaced using setattr(), allowing LoRA adapters to be inserted without modifying the overall transformer architecture. This mirrors the core idea of the original LoRA paper: augment existing layers with trainable low-rank adapters while preserving the pre-trained weights.

In [11]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Load a lightweight pre-trained model (GPT-2 is perfect for a free T4 GPU tier)
model_name = "gpt2"
print(f"Loading {model_name}...")
base_model = AutoModelForCausalLM.from_pretrained(model_name).cuda()

# 2. Function to recursively replace standard Conv1D/Linear layers with our Custom LoRA
def apply_lora_surgery(model, target_modules=["c_attn"], r=4, lora_alpha=8):
    """
    Scans the model architecture and replaces specific target layers with CustomLoRALinear.
    In GPT-2, the attention layers are stored as 'c_attn' (Hugging Face Conv1D layers).
    """
    for name, module in model.named_children():
        # If the child module contains sub-modules, recurse down the tree
        if len(list(module.children())) > 0:
            apply_lora_surgery(module, target_modules, r, lora_alpha)

        # Check if this module matches one of our target injection sites
        if name in target_modules:
            # For GPT-2's c_attn, out_features is the combined Q, K, V dimensions (3 * embed_dim)
            in_features = module.weight.shape[0]
            out_features = module.weight.shape[1]

            # Create our custom LoRA layer
            lora_layer = CustomLoRALinear(in_features, out_features, r=r, lora_alpha=lora_alpha).cuda()

            # (Optional) For strict math mapping, copy the original pre-trained weights over
            # Since GPT-2 uses transposed Conv1D weights, we align them to standard linear weights
            with torch.no_grad():
                lora_layer.linear.weight.copy_(module.weight.t())
                if module.bias is not None:
                    lora_layer.linear.bias.copy_(module.bias)

            # Complete the surgery by overwriting the old module on the parent class
            setattr(model, name, lora_layer)

# Run the surgery
apply_lora_surgery(base_model, target_modules=["c_attn"], r=4, lora_alpha=8)
print("\nSurgery complete! Let's verify trainable parameters...")

# 3. Calculate and verify parameter counts
total_params = 0
trainable_params = 0

for name, param in base_model.named_parameters():
    total_params += param.numel()
    if param.requires_grad:
        trainable_params += param.numel()

print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters (LoRA weights only): {trainable_params:,}")
print(f"Percentage of model being trained: {(trainable_params / total_params) * 100:.4f}%")

Loading gpt2...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Surgery complete! Let's verify trainable parameters...
Total Parameters: 124,587,264
Trainable Parameters (LoRA weights only): 103,353,600
Percentage of model being trained: 82.9568%


5. Injecting LoRA into GPT-2

After validating the custom layer in isolation, the next step is to integrate it into a pre-trained transformer.

The GPT-2 language model is first loaded from the Hugging Face Transformers library. Every parameter in the original model is frozen by setting requires_grad=False, ensuring that no updates are applied to the pre-trained weights during optimization.

LoRA modules are then injected into the target attention projection layers (c_attn) using the custom surgery function. Rather than replacing the entire model, the surgery selectively augments existing linear layers with low-rank adapters while preserving the original architecture.

Finally, the notebook computes the total number of parameters and the number of trainable parameters after surgery. This serves as empirical verification that only a very small fraction of the model will be updated during fine-tuning, demonstrating the parameter efficiency that motivates LoRA.

In [15]:
from transformers import AutoModelForCausalLM

# 1. Load the model
model_name = "gpt2"
print(f"Loading {model_name}...")
base_model = AutoModelForCausalLM.from_pretrained(model_name).cuda()

# === THE FIX: Freeze EVERY parameter in the base model upfront ===
for param in base_model.parameters():
    param.requires_grad = False

# 2. Run the surgery exactly like before
apply_lora_surgery(base_model, target_modules=["c_attn"], r=4, lora_alpha=8)
print("\nSurgery complete! Let's verify trainable parameters...")

# 3. Recalculate parameter counts
total_params = 0
trainable_params = 0

for name, param in base_model.named_parameters():
    total_params += param.numel()
    if param.requires_grad:
        trainable_params += param.numel()

print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters (LoRA weights only): {trainable_params:,}")
print(f"Percentage of model being trained: {(trainable_params / total_params) * 100:.4f}%")

Loading gpt2...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Surgery complete! Let's verify trainable parameters...
Total Parameters: 124,587,264
Trainable Parameters (LoRA weights only): 175,104
Percentage of model being trained: 0.1405%


6. Verifying Gradient Flow During Fine-Tuning

The final experiment performs a complete forward and backward pass using real text inputs tokenized with the GPT-2 tokenizer.

A standard AdamW optimizer is configured to update only parameters marked as trainable. Because all original GPT-2 weights have been frozen, the optimizer should receive only the LoRA matrices.

After computing the language modeling loss and performing backpropagation, gradients are inspected throughout the network.

The expected behavior is that:

* LoRA matrices receive gradients
* frozen pre-trained weights receive no gradients.

This provides direct evidence that optimization occurs exclusively through the low-rank adapters, confirming that the custom implementation follows the intended parameter-efficient training strategy.

In [14]:
import torch.optim as optim

# 1. Initialize a real tokenizer for GPT-2 to get actual text token inputs
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# 2. Create actual text inputs and generate a batch
text = ["Fine-tuning transformers from scratch feels incredible.", "LoRA math works perfectly."]
inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to("cuda")

# 3. Setup a standard optimizer tracking ONLY the trainable parameters
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, base_model.parameters()), lr=1e-4)

# 4. Forward Pass
outputs = base_model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"], labels=inputs["input_ids"])
loss = outputs.loss

# 5. Backward Pass
optimizer.zero_grad()
loss.backward()

# 6. EXPLICIT PROOF: Verify gradients exist ONLY on LoRA weights
print(f"Current Loss: {loss.item():.4f}\n")
print("Checking gradients across layers...")

lora_grad_found = False
base_grad_found = False

for name, param in base_model.named_parameters():
    if "lora_" in name:
        if param.grad is not None:
            lora_grad_found = True
    elif "linear.weight" in name or "wte" in name: # original weights
        if param.grad is not None:
            base_grad_found = True

print(f"Did our Custom LoRA weights receive gradients? -> {lora_grad_found}")
print(f"Did the frozen pre-trained weights receive gradients? -> {base_grad_found}")

Current Loss: 6.3205

Checking gradients across layers...
Did our Custom LoRA weights receive gradients? -> True
Did the frozen pre-trained weights receive gradients? -> False


Conclusion

In this notebook, we reconstructed Microsoft's LoRA algorithm from scratch and integrated it into GPT-2 without relying on PEFT or the official implementation. Through parameter counting and gradient inspection, we verified that optimization occurs exclusively through the low-rank adapters while the original transformer remains frozen. This implementation serves as a foundation for future experiments, including rank ablations, weight merging, and QLoRA.